# Análisis Diferencial: NPCR y UACI

Valida qué tan sensible es el cifrado (Pauli + autómata celular) a un cambio mínimo en la imagen de entrada. Reutiliza las mismas funciones de `encrypt.ipynb`, empaquetadas en `cifrar_imagen_completa`, para poder cifrar dos veces (original y con 1 píxel modificado) y comparar los resultados.

In [1]:
%pip install numpy
%pip install pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
from PIL import Image
from pathlib import Path

# Importaciones directas desde tus modulos
from pipeline import cifrar_imagen_completa
from io_imagen import obtener_rgb, filas_a_matriz_pixeles
from metricas import npcr, uaci

# Configuración de rutas
base = Path().resolve().parent
ruta_imagen = base / "Imagenes" / "Prueba2.png"

## Pipeline completo empaquetado

Junta todos los pasos de `encrypt.ipynb` (generación de llave → cifrado Pauli → separación en capas → cifrado con autómata celular) en una sola función. Devuelve las matrices enteras cifradas (mismo alto x ancho que la imagen original) — son las que realmente hay que comparar, no la versión normalizada que se guarda solo para visualización en el TIFF.

## Funciones NPCR y UACI

In [3]:
def npcr(c1, c2):
    """Number of Pixels Change Rate (%). 100% = todos los píxeles cambiaron."""
    c1 = np.asarray(c1)
    c2 = np.asarray(c2)
    assert c1.shape == c2.shape, "Las imágenes cifradas deben tener el mismo tamaño"
    D = (c1 != c2).astype(np.float64)
    return 100.0 * D.sum() / D.size

def uaci(c1, c2, valor_max):
    """Unified Average Changing Intensity (%). valor_max = máximo valor
    representable con el número de bits usado en el cifrado (2**num_bits - 1)."""
    c1 = np.asarray(c1, dtype=np.float64)
    c2 = np.asarray(c2, dtype=np.float64)
    assert c1.shape == c2.shape
    return 100.0 * np.sum(np.abs(c1 - c2)) / (c1.size * valor_max)

## Ejecutar el análisis diferencial

1. Carga la imagen original.
2. Genera una copia idéntica con **un solo píxel modificado** (+1 en el canal R de la esquina superior izquierda, con desbordamiento cíclico 255→0).
3. Cifra ambas con el pipeline completo.
4. Calcula NPCR y UACI por canal.

In [4]:
# 1. Imagen original
R, G, B = obtener_rgb(str(ruta_imagen))

# 2. Copia con un solo píxel modificado
img_original = Image.open(str(ruta_imagen)).convert("RGB")
arr_mod = np.array(img_original)

fila, col = 0, 0  # posición del píxel a modificar
arr_mod[fila, col, 0] = (int(arr_mod[fila, col, 0]) + 1) % 256  # +1 en canal R

img_mod = Image.fromarray(arr_mod, mode="RGB")
R2, G2, B2 = img_mod.split()

# 3. Cifrar ambas versiones con el pipeline empaquetado
cifrado1 = cifrar_imagen_completa(R, G, B)
cifrado2 = cifrar_imagen_completa(R2, G2, B2)

# 4. Reconstruir matrices enteras usando la utilidad de io_imagen
R1_mat = filas_a_matriz_pixeles(cifrado1.filas_R_cif, cifrado1.num_bits, cifrado1.w)
G1_mat = filas_a_matriz_pixeles(cifrado1.filas_G_cif, cifrado1.num_bits, cifrado1.w)
B1_mat = filas_a_matriz_pixeles(cifrado1.filas_B_cif, cifrado1.num_bits, cifrado1.w)

R2_mat = filas_a_matriz_pixeles(cifrado2.filas_R_cif, cifrado2.num_bits, cifrado2.w)
G2_mat = filas_a_matriz_pixeles(cifrado2.filas_G_cif, cifrado2.num_bits, cifrado2.w)
B2_mat = filas_a_matriz_pixeles(cifrado2.filas_B_cif, cifrado2.num_bits, cifrado2.w)

nb1, nb2 = cifrado1.num_bits, cifrado2.num_bits

if nb1 != nb2:
    print(f"Aviso: la profundidad de bits difiere entre corridas ({nb1} vs {nb2} bits). "
          "Es normal por la sensibilidad del cifrado; se usa el mayor de los dos para normalizar UACI.\n")

valor_max = (2 ** max(nb1, nb2)) - 1

# 5. Calcular NPCR y UACI por canal
resultados = {}
for canal, c1, c2 in [("R", R1_mat, R2_mat), ("G", G1_mat, G2_mat), ("B", B1_mat, B2_mat)]:
    resultados[canal] = {
        "NPCR": npcr(c1, c2),
        "UACI": uaci(c1, c2, valor_max),
    }

print(f"{'Canal':<6}{'NPCR (%)':<14}{'UACI (%)':<14}")
for canal, m in resultados.items():
    print(f"{canal:<6}{m['NPCR']:<14.4f}{m['UACI']:<14.4f}")

npcr_prom = np.mean([resultados[c]["NPCR"] for c in "RGB"])
uaci_prom = np.mean([resultados[c]["UACI"] for c in "RGB"])
print(f"\nPromedio -> NPCR = {npcr_prom:.4f}%   UACI = {uaci_prom:.4f}%")
print("Referencia ideal:      NPCR ≈ 99.6094%   UACI ≈ 33.4635%")

Aviso: la profundidad de bits difiere entre corridas (13 vs 12 bits). Es normal por la sensibilidad del cifrado; se usa el mayor de los dos para normalizar UACI.

Canal NPCR (%)      UACI (%)      
R     99.9840       32.7382       
G     99.9889       32.5004       
B     99.9832       31.7731       

Promedio -> NPCR = 99.9854%   UACI = 32.3372%
Referencia ideal:      NPCR ≈ 99.6094%   UACI ≈ 33.4635%


In [5]:
# 1. Imagen original
R, G, B = obtener_rgb(str(ruta_imagen))

# 2. Copia con un solo píxel modificado
img_original = Image.open(str(ruta_imagen)).convert("RGB")
arr_mod = np.array(img_original)

fila, col = 0, 0  # posición del píxel a modificar
arr_mod[fila, col, 0] = (int(arr_mod[fila, col, 0]) + 1) % 256  # +1 en canal R

img_mod = Image.fromarray(arr_mod, mode="RGB")
R2, G2, B2 = img_mod.split()

# 3. Cifrar ambas versiones con el pipeline completo
cifrado1 = cifrar_imagen_completa(R, G, B)
cifrado2 = cifrar_imagen_completa(R2, G2, B2)

R1_cif, G1_cif, B1_cif, nb1 = cifrado1.filas_R_cif, cifrado1.filas_G_cif, cifrado1.filas_B_cif, cifrado1.num_bits
R2_cif, G2_cif, B2_cif, nb2 = cifrado2.filas_R_cif, cifrado2.filas_G_cif, cifrado2.filas_B_cif, cifrado2.num_bits

if nb1 != nb2:
    print(f"Aviso: la profundidad de bits difiere entre corridas ({nb1} vs {nb2} bits). "
          "Es normal por la sensibilidad del cifrado; se usa el mayor de los dos para normalizar UACI.\n")

valor_max = (2 ** max(nb1, nb2)) - 1

# 4. NPCR y UACI por canal
resultados = {}
for canal, c1, c2 in [("R", R1_cif, R2_cif), ("G", G1_cif, G2_cif), ("B", B1_cif, B2_cif)]:
    resultados[canal] = {
        "NPCR": npcr(c1, c2),
        "UACI": uaci(c1, c2, valor_max),
    }

print(f"{'Canal':<6}{'NPCR (%)':<14}{'UACI (%)':<14}")
for canal, m in resultados.items():
    print(f"{canal:<6}{m['NPCR']:<14.4f}{m['UACI']:<14.4f}")

npcr_prom = np.mean([resultados[c]["NPCR"] for c in "RGB"])
uaci_prom = np.mean([resultados[c]["UACI"] for c in "RGB"])
print(f"\nPromedio -> NPCR = {npcr_prom:.4f}%   UACI = {uaci_prom:.4f}%")
print("Referencia ideal:      NPCR ≈ 99.6094%   UACI ≈ 33.4635%")

Aviso: la profundidad de bits difiere entre corridas (13 vs 12 bits). Es normal por la sensibilidad del cifrado; se usa el mayor de los dos para normalizar UACI.



AssertionError: Las imágenes cifradas deben tener el mismo tamaño

Prueba robusta: promedio sobre varias perturbaciones

Un solo píxel modificado puede dar resultados algo ruidosos. Para un análisis más riguroso

In [ ]:
def analisis_diferencial_promedio(ruta_imagen, n_pruebas=10, semilla=42):
    rng = np.random.default_rng(semilla)
    img_original = Image.open(str(ruta_imagen)).convert("RGB")
    R, G, B = img_original.split()

    cifrado1 = cifrar_imagen_completa(R, G, B)
    nb1 = cifrado1.num_bits
    R1_cif = filas_a_matriz_pixeles(cifrado1.filas_R_cif, nb1, cifrado1.w)
    G1_cif = filas_a_matriz_pixeles(cifrado1.filas_G_cif, nb1, cifrado1.w)
    B1_cif = filas_a_matriz_pixeles(cifrado1.filas_B_cif, nb1, cifrado1.w)

    npcr_vals = {"R": [], "G": [], "B": []}
    uaci_vals = {"R": [], "G": [], "B": []}

    h, w = img_original.height, img_original.width
    for _ in range(n_pruebas):
        arr_mod = np.array(img_original)
        fila = rng.integers(0, h)
        col = rng.integers(0, w)
        canal_idx = rng.integers(0, 3)
        arr_mod[fila, col, canal_idx] = (int(arr_mod[fila, col, canal_idx]) + 1) % 256

        img_mod = Image.fromarray(arr_mod, mode="RGB")
        R2, G2, B2 = img_mod.split()
        cifrado2 = cifrar_imagen_completa(R2, G2, B2)
        nb2 = cifrado2.num_bits
        R2_cif = filas_a_matriz_pixeles(cifrado2.filas_R_cif, nb2, cifrado2.w)
        G2_cif = filas_a_matriz_pixeles(cifrado2.filas_G_cif, nb2, cifrado2.w)
        B2_cif = filas_a_matriz_pixeles(cifrado2.filas_B_cif, nb2, cifrado2.w)

        valor_max = (2 ** max(nb1, nb2)) - 1
        for canal, c1, c2 in [("R", R1_cif, R2_cif), ("G", G1_cif, G2_cif), ("B", B1_cif, B2_cif)]:
            npcr_vals[canal].append(npcr(c1, c2))
            uaci_vals[canal].append(uaci(c1, c2, valor_max))

    print(f"Promedio sobre {n_pruebas} pruebas:")
    print(f"{'Canal':<6}{'NPCR (%)':<14}{'UACI (%)':<14}")
    for canal in "RGB":
        print(f"{canal:<6}{np.mean(npcr_vals[canal]):<14.4f}{np.mean(uaci_vals[canal]):<14.4f}")

analisis_diferencial_promedio(ruta_imagen, n_pruebas=10)